# IPL Match Classification - Grandmaster Competition Pipeline

**Objective:** Predict match outcomes across 4 classes (A_small, A_big, B_small, B_big) with calibrated ensemble models targeting Log Loss 0.95-1.10

**Architecture:**
- Historical feature engineering engine
- Calibrated XGBoost + LightGBM ensemble
- Optuna hyperparameter optimization (30+ trials)
- Strict probability normalization for submission

## Cell 1: Imports & Configuration

In [1]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
import xgboost as xgb
import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import log_loss
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
import gc

warnings.filterwarnings('ignore')
np.random.seed(42)

DATA_PATH = Path('.')
print("✓ All imports successful")

✓ All imports successful


## Cell 2: Load Raw Data

In [2]:
train_bbb = pd.read_csv(DATA_PATH / 'train_IPL.csv')
public_lb = pd.read_csv(DATA_PATH / 'public_lb_matches.csv')
schedule = pd.read_csv(DATA_PATH / 'schedule.csv')
sample_sub = pd.read_csv(DATA_PATH / 'sample_submission.csv')

print(f"Train ball-by-ball: {train_bbb.shape}")
print(f"Public LB matches: {public_lb.shape}")
print(f"Schedule: {schedule.shape}")
print(f"Sample submission: {sample_sub.shape}")
print(f"\nTarget classes in submission: {sample_sub.columns[1:].tolist()}")

Train ball-by-ball: (272704, 38)
Public LB matches: (48, 9)
Schedule: (5, 6)
Sample submission: (53, 5)

Target classes in submission: ['A_small', 'A_big', 'B_small', 'B_big']


## Cell 3: Team Name Cleaning & Standardization

In [3]:
TEAM_NAME_MAPPING = {
    'Kolkata Knight Riders': 'Kolkata Knight Riders',
    'Royal Challengers Bangalore': 'Royal Challengers Bengaluru',
    'Royal Challengers Bengaluru': 'Royal Challengers Bengaluru',
    'Mumbai Indians': 'Mumbai Indians',
    'Chennai Super Kings': 'Chennai Super Kings',
    'Rajasthan Royals': 'Rajasthan Royals',
    'Delhi Daredevils': 'Delhi Capitals',
    'Delhi Capitals': 'Delhi Capitals',
    'Sunrisers Hyderabad': 'Sunrisers Hyderabad',
    'Hyderabad': 'Sunrisers Hyderabad',
    'Kings XI Punjab': 'Kings XI Punjab',
    'Punjab Kings': 'Punjab Kings',
    'Kochi Tuskers Kerala': 'Kochi Tuskers Kerala',
    'Pune Warriors': 'Pune Warriors',
    'Gujarat Lions': 'Gujarat Lions',
    'Rising Pune Supergiants': 'Rising Pune Supergiants',
    'Deccan Chargers': 'Deccan Chargers',
    'Bangalore': 'Royal Challengers Bengaluru',
    'Gujarat Titans': 'Gujarat Titans',
    'Lucknow Super Giants': 'Lucknow Super Giants',
}

def clean_team_name(name):
    if pd.isna(name):
        return name
    return TEAM_NAME_MAPPING.get(str(name).strip(), str(name).strip())

for col in ['Bat First', 'Bat Second', 'toss_winner', 'match_won_by']:
    if col in train_bbb.columns:
        train_bbb[col] = train_bbb[col].apply(clean_team_name)

for col in ['Team A', 'Team B', 'Toss Winner']:
    if col in public_lb.columns:
        public_lb[col] = public_lb[col].apply(clean_team_name)

for col in schedule.columns:
    if 'team' in col.lower():
        schedule[col] = schedule[col].apply(clean_team_name)

print("✓ Team names standardized")

✓ Team names standardized


## Cell 4: Aggregate Ball-by-Ball to Match Level

In [4]:
def create_train_labels(train_bbb):
    match_groups = train_bbb.groupby('Match ID').agg({
        'Bat First': 'first',
        'Bat Second': 'first',
        'match_won_by': 'first',
        'result_type': 'first',
        'Venue': 'first',
        'city': 'first',
        'Date': 'first',
        'toss_winner': 'first',
        'toss_decision': 'first',
        'season': 'first'
    }).reset_index()
    
    match_groups.rename(columns={'Bat First': 'Team A', 'Bat Second': 'Team B'}, inplace=True)
    
    def assign_label(row):
        winner = row['match_won_by']
        result_type = row['result_type']
        
        if pd.isna(winner) or pd.isna(result_type):
            return None
        
        winner_is_team_a = (winner == row['Team A'])
        
        if 'wicket' in str(result_type).lower():
            return 'A_small' if winner_is_team_a else 'B_small'
        else:
            return 'A_big' if winner_is_team_a else 'B_big'
    
    match_groups['target'] = match_groups.apply(assign_label, axis=1)
    match_groups = match_groups[match_groups['target'].notna()].copy()
    
    return match_groups

train_matches = create_train_labels(train_bbb)
print(f"Training matches: {len(train_matches)}")
print(f"Target distribution:\n{train_matches['target'].value_counts()}")

Training matches: 21
Target distribution:
target
B_big    21
Name: count, dtype: int64


## Cell 5: Build Historical Feature Engine

In [5]:
class HistoricalFeatureEngine:
    def __init__(self, train_matches):
        self.train_matches = train_matches.copy()
        self._build_features()
    
    def _build_features(self):
        self.overall_wins = {}
        self.overall_games = {}
        self.bat_first_wins = {}
        self.bat_first_games = {}
        self.bat_second_wins = {}
        self.bat_second_games = {}
        self.venue_avg_runs = {}
        self.venue_games = {}
        
        for _, row in self.train_matches.iterrows():
            team_a, team_b = row['Team A'], row['Team B']
            venue = row['Venue']
            target = row['target']
            
            is_a_win = target in ['A_small', 'A_big']
            margin_type = 'small' if 'small' in target else 'big'
            
            for team in [team_a, team_b]:
                if team not in self.overall_wins:
                    self.overall_wins[team] = 0
                    self.overall_games[team] = 0
                    self.bat_first_wins[team] = 0
                    self.bat_first_games[team] = 0
                    self.bat_second_wins[team] = 0
                    self.bat_second_games[team] = 0
                
                is_winner = (team == team_a and is_a_win) or (team == team_b and not is_a_win)
                self.overall_games[team] += 1
                if is_winner:
                    self.overall_wins[team] += 1
                
                is_bat_first = (team == team_a)
                if is_bat_first:
                    self.bat_first_games[team] += 1
                    if is_winner:
                        self.bat_first_wins[team] += 1
                else:
                    self.bat_second_games[team] += 1
                    if is_winner:
                        self.bat_second_wins[team] += 1
            
            if venue not in self.venue_avg_runs:
                self.venue_avg_runs[venue] = 0
                self.venue_games[venue] = 0
            self.venue_games[venue] += 1
            self.venue_avg_runs[venue] += 150
    
    def get_features(self, team_a, team_b, venue, toss_winner):
        features = {}
        
        a_overall_wins = self.overall_wins.get(team_a, 0)
        a_overall_games = self.overall_games.get(team_a, 1)
        features['team_a_win_rate'] = a_overall_wins / max(a_overall_games, 1)
        
        b_overall_wins = self.overall_wins.get(team_b, 0)
        b_overall_games = self.overall_games.get(team_b, 1)
        features['team_b_win_rate'] = b_overall_wins / max(b_overall_games, 1)
        
        a_bat_first_wins = self.bat_first_wins.get(team_a, 0)
        a_bat_first_games = self.bat_first_games.get(team_a, 1)
        features['team_a_bat_first_wr'] = a_bat_first_wins / max(a_bat_first_games, 1)
        
        b_bat_second_wins = self.bat_second_wins.get(team_b, 0)
        b_bat_second_games = self.bat_second_games.get(team_b, 1)
        features['team_b_bat_second_wr'] = b_bat_second_wins / max(b_bat_second_games, 1)
        
        features['win_rate_diff'] = features['team_a_win_rate'] - features['team_b_win_rate']
        features['toss_a_wins'] = 1 if toss_winner == team_a else 0
        
        venue_avg = self.venue_avg_runs.get(venue, 150) / max(self.venue_games.get(venue, 1), 1)
        features['venue_avg_runs'] = venue_avg
        
        features['team_a_overall_games'] = a_overall_games
        features['team_b_overall_games'] = b_overall_games
        
        return features

feature_engine = HistoricalFeatureEngine(train_matches)
print("✓ Historical feature engine built")

✓ Historical feature engine built


## Cell 6: Create Training Feature Matrix

In [6]:
def create_feature_matrix(matches_df, feature_engine, label_encoders=None):
    X_list = []
    feature_names = None
    
    for _, row in matches_df.iterrows():
        features = feature_engine.get_features(
            row['Team A'], 
            row['Team B'], 
            row['Venue'], 
            row['toss_winner']
        )
        X_list.append(features)
    
    X = pd.DataFrame(X_list)
    
    categorical_cols = ['Venue']
    if label_encoders is None:
        label_encoders = {}
    
    for col in categorical_cols:
        if col not in label_encoders:
            label_encoders[col] = LabelEncoder()
            matches_df[f'{col}_encoded'] = label_encoders[col].fit_transform(matches_df[col].astype(str))
        else:
            matches_df[f'{col}_encoded'] = label_encoders[col].transform(matches_df[col].astype(str))
        X[f'{col}_encoded'] = matches_df[f'{col}_encoded'].values
    
    return X, label_encoders

X_train, label_encoders = create_feature_matrix(train_matches, feature_engine)
y_train = train_matches['target'].values

print(f"Training features shape: {X_train.shape}")
print(f"Feature columns: {X_train.columns.tolist()}")
print(f"\nFirst row features:\n{X_train.iloc[0]}")

Training features shape: (21, 10)
Feature columns: ['team_a_win_rate', 'team_b_win_rate', 'team_a_bat_first_wr', 'team_b_bat_second_wr', 'win_rate_diff', 'toss_a_wins', 'venue_avg_runs', 'team_a_overall_games', 'team_b_overall_games', 'Venue_encoded']

First row features:
team_a_win_rate           0.4
team_b_win_rate           0.6
team_a_bat_first_wr       0.0
team_b_bat_second_wr      1.0
win_rate_diff            -0.2
toss_a_wins               0.0
venue_avg_runs          150.0
team_a_overall_games      5.0
team_b_overall_games      5.0
Venue_encoded             7.0
Name: 0, dtype: float64


## Cell 7: Prepare Test Data

In [7]:
test_matches = public_lb.copy()
test_matches.rename(columns={'Team A': 'Team A', 'Team B': 'Team B'}, inplace=True)

if 'Venue' not in test_matches.columns:
    test_matches['Venue'] = 'Unknown'
if 'Toss Winner' not in test_matches.columns:
    test_matches['Toss Winner'] = test_matches['Team A']

test_matches.rename(columns={'Toss Winner': 'toss_winner'}, inplace=True)

X_test_list = []
for _, row in test_matches.iterrows():
    features = feature_engine.get_features(
        row['Team A'],
        row['Team B'],
        row.get('Venue', 'Unknown'),
        row.get('toss_winner', row['Team A'])
    )
    X_test_list.append(features)

X_test = pd.DataFrame(X_test_list)

for col in ['Venue']:
    if col in label_encoders:
        test_matches[f'{col}_encoded'] = label_encoders[col].transform(
            test_matches[col].fillna('Unknown').astype(str)
        )
        X_test[f'{col}_encoded'] = test_matches[f'{col}_encoded'].values

print(f"Test features shape: {X_test.shape}")
print(f"Test match IDs: {test_matches.shape[0]}")

KeyError: 'Team A'

## Cell 8: Initialize Calibrated Model Architecture

In [8]:
from sklearn.preprocessing import StandardScaler

label_le = LabelEncoder()
y_train_encoded = label_le.fit_transform(y_train)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Classes: {label_le.classes_}")
print(f"Encoded target: {np.unique(y_train_encoded)}")
print(f"✓ Data prepared and scaled")

NameError: name 'X_test' is not defined

## Cell 9: Define Objective Function for Optuna

In [ ]:
def objective(trial):
    xgb_max_depth = trial.suggest_int('xgb_max_depth', 4, 12)
    xgb_learning_rate = trial.suggest_float('xgb_learning_rate', 0.01, 0.3, log=True)
    xgb_subsample = trial.suggest_float('xgb_subsample', 0.6, 0.95)
    xgb_colsample = trial.suggest_float('xgb_colsample', 0.6, 0.95)
    xgb_gamma = trial.suggest_float('xgb_gamma', 0, 5)
    xgb_lambda = trial.suggest_float('xgb_lambda', 0.5, 10)
    
    lgb_num_leaves = trial.suggest_int('lgb_num_leaves', 15, 100)
    lgb_learning_rate = trial.suggest_float('lgb_learning_rate', 0.01, 0.3, log=True)
    lgb_subsample = trial.suggest_float('lgb_subsample', 0.6, 0.95)
    lgb_colsample = trial.suggest_float('lgb_colsample', 0.6, 0.95)
    lgb_lambda_l1 = trial.suggest_float('lgb_lambda_l1', 0, 5)
    lgb_lambda_l2 = trial.suggest_float('lgb_lambda_l2', 0, 5)
    
    xgb_model = xgb.XGBClassifier(
        max_depth=xgb_max_depth,
        learning_rate=xgb_learning_rate,
        subsample=xgb_subsample,
        colsample_bytree=xgb_colsample,
        gamma=xgb_gamma,
        reg_lambda=xgb_lambda,
        n_estimators=300,
        random_state=42,
        verbosity=0,
        use_label_encoder=False,
        eval_metric='mlogloss'
    )
    
    lgb_model = lgb.LGBMClassifier(
        num_leaves=lgb_num_leaves,
        learning_rate=lgb_learning_rate,
        subsample=lgb_subsample,
        colsample_bytree=lgb_colsample,
        lambda_l1=lgb_lambda_l1,
        lambda_l2=lgb_lambda_l2,
        n_estimators=300,
        random_state=42,
        verbose=-1
    )
    
    xgb_calib = CalibratedClassifierCV(
        estimator=xgb_model,
        method='isotonic',
        cv=5
    )
    
    lgb_calib = CalibratedClassifierCV(
        estimator=lgb_model,
        method='isotonic',
        cv=5
    )
    
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    
    for train_idx, val_idx in skf.split(X_train_scaled, y_train_encoded):
        X_tr, X_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
        y_tr, y_val = y_train_encoded[train_idx], y_train_encoded[val_idx]
        
        xgb_calib.fit(X_tr, y_tr)
        lgb_calib.fit(X_tr, y_tr)
        
        xgb_proba = xgb_calib.predict_proba(X_val)
        lgb_proba = lgb_calib.predict_proba(X_val)
        
        ensemble_proba = 0.5 * xgb_proba + 0.5 * lgb_proba
        ensemble_proba = np.clip(ensemble_proba, 1e-7, 1 - 1e-7)
        
        val_loss = log_loss(y_val, ensemble_proba)
        scores.append(val_loss)
    
    mean_score = np.mean(scores)
    return mean_score

print("✓ Objective function defined")

## Cell 10: Run Optuna Hyperparameter Optimization

In [ ]:
sampler = TPESampler(seed=42)
pruner = MedianPruner(n_warmup_steps=10)

study = optuna.create_study(
    direction='minimize',
    sampler=sampler,
    pruner=pruner
)

print("Starting Optuna optimization (40 trials)...")
study.optimize(objective, n_trials=40, show_progress_bar=True)

print(f"\n✓ Best trial log loss: {study.best_value:.6f}")
print(f"Best hyperparameters:\n{study.best_params}")

best_params = study.best_params

## Cell 11: Train Final Models with Best Hyperparameters

In [ ]:
final_xgb = xgb.XGBClassifier(
    max_depth=best_params['xgb_max_depth'],
    learning_rate=best_params['xgb_learning_rate'],
    subsample=best_params['xgb_subsample'],
    colsample_bytree=best_params['xgb_colsample'],
    gamma=best_params['xgb_gamma'],
    reg_lambda=best_params['xgb_lambda'],
    n_estimators=300,
    random_state=42,
    verbosity=0,
    use_label_encoder=False,
    eval_metric='mlogloss'
)

final_lgb = lgb.LGBMClassifier(
    num_leaves=best_params['lgb_num_leaves'],
    learning_rate=best_params['lgb_learning_rate'],
    subsample=best_params['lgb_subsample'],
    colsample_bytree=best_params['lgb_colsample'],
    lambda_l1=best_params['lgb_lambda_l1'],
    lambda_l2=best_params['lgb_lambda_l2'],
    n_estimators=300,
    random_state=42,
    verbose=-1
)

final_xgb_calib = CalibratedClassifierCV(
    estimator=final_xgb,
    method='isotonic',
    cv=5
)

final_lgb_calib = CalibratedClassifierCV(
    estimator=final_lgb,
    method='isotonic',
    cv=5
)

print("Training calibrated XGBoost...")
final_xgb_calib.fit(X_train_scaled, y_train_encoded)

print("Training calibrated LightGBM...")
final_lgb_calib.fit(X_train_scaled, y_train_encoded)

print("✓ Final models trained and calibrated")

## Cell 12: Generate Predictions on Test Set

In [ ]:
print("Generating predictions...")

xgb_proba = final_xgb_calib.predict_proba(X_test_scaled)
lgb_proba = final_lgb_calib.predict_proba(X_test_scaled)

ensemble_proba = 0.5 * xgb_proba + 0.5 * lgb_proba
ensemble_proba = np.clip(ensemble_proba, 1e-7, 1 - 1e-7)

print(f"Ensemble predictions shape: {ensemble_proba.shape}")
print(f"Sample prediction (first row):\n{ensemble_proba[0]}")
print(f"Sample sum: {ensemble_proba[0].sum():.10f}")

## Cell 13: Create Submission with Probability Normalization

In [ ]:
submission = pd.DataFrame()
submission['ID'] = test_matches.index if 'ID' not in test_matches.columns else test_matches['ID'].values

class_mapping = {i: cls for i, cls in enumerate(label_le.classes_)}

for class_idx, class_name in class_mapping.items():
    submission[class_name] = ensemble_proba[:, class_idx]

def normalize_probabilities_row(row):
    prob_cols = ['A_small', 'A_big', 'B_small', 'B_big']
    
    first_three_sum = 0
    for col in prob_cols[:3]:
        rounded_val = round(row[col], 2)
        row[col] = rounded_val
        first_three_sum += rounded_val
    
    row[prob_cols[3]] = round(1.0 - first_three_sum, 2)
    row[prob_cols[3]] = max(0.00, min(1.00, row[prob_cols[3]]))
    
    actual_sum = sum(row[col] for col in prob_cols)
    if abs(actual_sum - 1.0) > 0.001:
        diff = 1.0 - actual_sum
        row[prob_cols[3]] = round(row[prob_cols[3]] + diff, 2)
    
    return row

submission = submission.apply(normalize_probabilities_row, axis=1)

print(f"Submission shape: {submission.shape}")
print(f"\nFirst 5 rows:\n{submission.head()}")
print(f"\nColumn sums (should all be 1.0):")
prob_cols = ['A_small', 'A_big', 'B_small', 'B_big']
for col in prob_cols:
    print(f"  {col}: min={submission[col].min():.2f}, max={submission[col].max():.2f}")

row_sums = submission[prob_cols].sum(axis=1)
print(f"\nRow sum statistics:")
print(f"  Min: {row_sums.min():.10f}")
print(f"  Max: {row_sums.max():.10f}")
print(f"  Mean: {row_sums.mean():.10f}")
print(f"  All equal 1.00? {(row_sums == 1.0).all()}")

## Cell 14: Save Submission

In [ ]:
submission_path = DATA_PATH / 'submission_fixed.csv'
submission.to_csv(submission_path, index=False)

print(f"✓ Submission saved to {submission_path}")
print(f"\nFile verification:")

verification = pd.read_csv(submission_path)
print(f"  Shape: {verification.shape}")
print(f"  Columns: {verification.columns.tolist()}")
print(f"  No NaN values: {not verification.isna().any().any()}")

verify_sums = verification[prob_cols].sum(axis=1)
print(f"  All rows sum to 1.0: {(verify_sums == 1.0).all()}")
print(f"\n✓ Competition pipeline complete!")
print(f"Expected Log Loss range: 0.95 - 1.10")